# RnD pipeline hauling lengkap pada Colab T4

Perbandingan video **45 detik pertama** dengan OCR nomor lambung, klasifikasi kendaraan,
crossing polygon, pengukuran kecepatan, overspeed, event pada final area, evidence crop,
laporan kendaraan dan video beranotasi. Kode bisnis diambil dari snapshot proyek hauling;
tidak memakai ulang hasil OCR tersimpan atau data deteksi cache.

| Varian | Detector kendaraan | Tracker | Model tambahan |
|---|---|---|---|
| bytetrack | PyTorch FP32 | ByteTrack | OCR dan classifier FP32 |
| botsort | PyTorch FP32 | BoT-SORT dengan sparseOptFlow GMC, tanpa ReID | Sama |
| deepsort | PyTorch FP32 | DeepSORT, n_init 3, buffer satu detik | MobileNetV2 ImageNet embedding FP32 |
| pytorch_fp16 | PyTorch FP16 | ByteTrack | Kontrol efek presisi; OCR dan classifier tetap FP32 |
| tensorrt_fp16 | TensorRT FP16 | ByteTrack | Hanya detector kendaraan diekspor; OCR dan classifier tetap FP32 |

BoT-SORT dan DeepSORT menggunakan implementasi/library yang disebutkan, bukan klaim identik dengan semua varian paper.
Embedding DeepSORT adalah baseline ImageNet, belum model ReID yang dilatih khusus kendaraan hauling.
TensorRT diuji sebagai backend detector dalam pipeline lengkap, bukan sebagai algoritma tracking keempat.

Notebook belum dieksekusi pada T4 oleh penyusunnya. Struktur notebook, pemetaan track,
pencocokan event dan kesetaraan event setelah instrumentasi diuji lokal.
Validasi lokal mencakup 27 pemeriksaan otomatis dan smoke test CPU: ByteTrack serta BoT-SORT
15 frame, DeepSORT 90 frame. DeepSORT menuntaskan satu event dengan OCR AEK0083,
speed 64,09 km/h dan status OVERSPEED, beserta report, evidence dan video tanpa error.
Ini adalah bukti integrasi, bukan benchmark performa atau akurasi T4.


## 1 Pilih runtime dan pasang dependency

Pilih **Runtime > Change runtime type > T4 GPU**. Gunakan runtime baru agar dependency lama tidak bercampur.
Versi PyTorch dan Ultralytics disamakan dengan proyek; wheel PyTorch yang dipasang adalah CUDA, bukan CPU.
TensorRT 10.x dipin karena exporter versi proyek memakai API 10.x. Restart session sesudah instalasi bila diminta.
Paket cukup besar; tunggu instalasi selesai. Runtime Python yang didukung paket pin adalah prasyarat;
bila Colab berubah dan wheel tidak tersedia, cocokkan stack sebelum melanjutkan.


In [ ]:
%pip install -q "torch==2.7.1" "torchvision==0.22.1" --index-url https://download.pytorch.org/whl/cu126
%pip install -q "ultralytics==8.3.228" "opencv-python==4.11.0.86" "numpy>=1.26,<3" "lap==0.5.12" "deep-sort-realtime==1.3.2" "setuptools==80.9.0" "tensorrt-cu12==10.13.3.9" "onnx>=1.12,<1.18" "onnxslim==0.1.71" "pandas>=2,<3" "pytest==8.4.2" "PyYAML>=6,<7"


In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import sys,json,hashlib,base64,io,zipfile,subprocess,shutil,time,gc
from pathlib import Path
import cv2,numpy as np,pandas as pd,torch,tensorrt as trt,ultralytics
assert torch.cuda.is_available(), 'Aktifkan GPU runtime; eksekusi CPU tidak diterima untuk benchmark T4.'
assert 'T4' in torch.cuda.get_device_name(0), torch.cuda.get_device_name(0)
assert torch.__version__.split('+')[0]=='2.7.1', 'Restart session setelah mengganti torch.'
assert ultralytics.__version__=='8.3.228'
assert trt.__version__.startswith('10.13.3'), trt.__version__
torch.backends.cudnn.benchmark=False
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
cv2.setNumThreads(1)
WORK=Path('/content/hauling_full_pipeline_rnd')
ROOT=WORK/'project'
ROOT.mkdir(parents=True,exist_ok=True)
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
env=dict(python=sys.version,torch=torch.__version__,torch_cuda=torch.version.cuda,
    ultralytics=ultralytics.__version__,tensorrt=trt.__version__,opencv=cv2.__version__,
    gpu=torch.cuda.get_device_name(0),compute_capability=torch.cuda.get_device_capability(0),
    gpu_info=subprocess.check_output(['nvidia-smi'],text=True),
    cpu=Path('/proc/cpuinfo').read_text().split('model name')[1].split('\n')[0].strip(' :'))
print(json.dumps(env,indent=2))


## 2 Ekstrak snapshot kode bawaan notebook

Kode pipeline sudah tertanam di notebook ini, sehingga tidak perlu akses repository atau token GitHub.
Hash source diverifikasi. Timing hooks hanya ditambahkan pada salinan modul di memori;
algoritma OCR, speed, pelanggaran dan pencatatan event tetap menggunakan source snapshot yang sama.


In [ ]:
SOURCE_B64='...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_B64))) as z:
    for name in z.namelist():
        target=(ROOT/name).resolve()
        assert target.is_relative_to(ROOT.resolve()), 'Unsafe archive path'
    z.extractall(ROOT)
code_manifest=json.loads((ROOT/'code_manifest.json').read_text())
for name,expected in code_manifest.items():
    assert sha256(ROOT/name)==expected,name
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
os.chdir(ROOT)
result=subprocess.run([sys.executable,'-m','pytest','tests/test_rnd_full_pipeline.py','-q'],cwd=ROOT,text=True)
assert result.returncode==0,'Uji adapter/timing gagal; hentikan sebelum benchmark.'
print('Snapshot kode dan regression checks OK')


## 3 Masukkan paket aset proyek

Gunakan file pendamping **Aset_RnD_Full_Pipeline_Colab.zip**. Paket berisi tujuh model asli,
video asli (yang diproses hanya 45 detik awal), kalibrasi offline dan referensi panel yang sama.
Konfigurasi hanya memuat field yang dibaca pipeline offline.

Pilih `ASSET_MODE='upload'` untuk upload ZIP dari komputer, atau `'drive'` lalu isi path ZIP di Drive.
Pengujian selalu mengekstrak ke disk lokal `/content`; tidak menjalankan inference langsung dari Drive.


In [ ]:
ASSET_MODE='upload'  # atau 'drive'
DRIVE_ASSET_ZIP='/content/drive/MyDrive/Aset_RnD_Full_Pipeline_Colab.zip'
EXPECTED_ASSET_SHA256='cb7d9cbe927070e68515e3e9ef7422f9a64c29310e7a62d591c75f954dab9f22'
if ASSET_MODE=='drive':
    from google.colab import drive
    drive.mount('/content/drive')
    archive=Path(DRIVE_ASSET_ZIP)
elif ASSET_MODE=='upload':
    from google.colab import files
    uploaded=files.upload()
    assert len(uploaded)==1,'Upload hanya ZIP aset pendamping.'
    name,blob=next(iter(uploaded.items()))
    archive=WORK/Path(name).name
    archive.write_bytes(blob)
    del uploaded,blob
    gc.collect()
else:raise ValueError('ASSET_MODE harus upload atau drive')
assert sha256(archive)==EXPECTED_ASSET_SHA256,'ZIP tidak cocok dengan snapshot notebook ini.'
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():assert (ROOT/name).resolve().is_relative_to(ROOT.resolve())
    z.extractall(ROOT)
asset_manifest=json.loads((ROOT/'assets_manifest.json').read_text())
for name,entry in asset_manifest.items():assert sha256(ROOT/name)==entry['sha256'],name
from hauling.config import read_config
cfg=read_config(ROOT/'model-requirement/camera_offline.conf')
from hauling.vision.hull_reference import ReferenceHullDetector
reference=ReferenceHullDetector(ROOT/'reference/panel_closeup.jpg')
VIDEO=ROOT/'video/20261007133257-3201.mp4'
cap=cv2.VideoCapture(str(VIDEO))
assert cap.isOpened()
FPS=cap.get(cv2.CAP_PROP_FPS)
assert FPS==25.0,FPS
assert (cap.get(cv2.CAP_PROP_FRAME_WIDTH),cap.get(cv2.CAP_PROP_FRAME_HEIGHT))==(1920.,1080.)
assert cap.get(cv2.CAP_PROP_FRAME_COUNT)>=1125
cap.release()
print('Aset OK. Kalibrasi:',cfg.distance_meter,'meter; limit:',cfg.speed_limit_kmh,'km/h; FPS:',FPS)


## 4 Tetapkan rancangan eksperimen

Default: 45 detik, 1.125 frame, lima konfigurasi, tiga pengulangan per konfigurasi.
Semua run menghasilkan CSV, report, foto event, hull crops, trace dan annotated.mp4;
beban output ini dimasukkan ke timing pipeline pada seluruh varian.

Set `REPEATS=1` untuk percobaan awal yang lebih singkat. Jangan menyebut satu run sebagai benchmark stabil.
Threshold kendaraan 0,25 dipertahankan dari proyek. Recovery confidence rendah belum teruji pada ambang ini.
Ukuran detector kendaraan dikontrol 640 x 640 (`rect=False`) pada semua backend;
ini berbeda dari kemungkinan letterbox persegi panjang pada runtime lama dan perlu dicatat saat membandingkan hasil historis.


In [ ]:
MAX_SECONDS=45
REPEATS=3
VARIANTS=['bytetrack','botsort','deepsort','pytorch_fp16','tensorrt_fp16']
SESSION=WORK/('results_'+time.strftime('%Y%m%d_%H%M%S'))
SESSION.mkdir(exist_ok=False)
(SESSION/'environment.json').write_text(json.dumps(env,indent=2))
shutil.copy2(ROOT/'code_manifest.json',SESSION/'code_manifest.json')
shutil.copy2(ROOT/'assets_manifest.json',SESSION/'assets_manifest.json')
(SESSION/'pip-freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
experiment=dict(video_seconds=MAX_SECONDS,frames=round(FPS*MAX_SECONDS),repeats=REPEATS,
    variants=VARIANTS,detector_shape=[640,640],rect=False,confidence=cfg.confidence,
    iou=cfg.iou,buffer_seconds=1.,ocr_sample_seconds=.2,ocr_lookahead_seconds=.3,
    event_trigger='first_final_area_contact',auxiliary_precision='FP32',
    artifacts_in_timed_run=['events.csv','annotated.mp4','hull_crops','report','detection_trace.jsonl'])
(SESSION/'experiment.json').write_text(json.dumps(experiment,indent=2))
print('Hasil disimpan di:',SESSION)


## 5 Preflight DeepSORT dan bangun TensorRT FP16

Model embedding MobileNetV2 dibundel oleh paket deep-sort-realtime dan memakai fitur ImageNet.
Preflight memastikan bobot tersedia sebelum pipeline memasuki mode offline.
Engine dibangun di T4 runtime ini; waktu build dilaporkan terpisah dari waktu pipeline.
Hanya detector kendaraan yang memakai TensorRT. Hull detector dan keempat model OCR tetap berjalan penuh di GPU lewat PyTorch.


In [ ]:
from deep_sort_realtime.deepsort_tracker import DeepSort
import deep_sort_realtime
embedder_weights=Path(deep_sort_realtime.__file__).parent/'embedder/weights/mobilenetv2_bottleneck_wts.pt'
assert embedder_weights.is_file()
probe=DeepSort(embedder='mobilenet',half=False,embedder_gpu=True,max_age=25,n_init=3)
probe.update_tracks([],frame=np.zeros((224,224,3),np.uint8))
env['deep_sort_realtime']='1.3.2'
env['deepsort_embedding_sha256']=sha256(embedder_weights)
(SESSION/'environment.json').write_text(json.dumps(env,indent=2))
del probe
gc.collect();torch.cuda.empty_cache()
from ultralytics import YOLO
export_dir=WORK/'engines';export_dir.mkdir(exist_ok=True)
export_pt=export_dir/'vehicle_fp16.pt'
shutil.copy2(ROOT/'model-requirement/Hauling_Vehicle_yolov8m_SingleCls_V11.pt',export_pt)
started=time.perf_counter()
exporter=YOLO(str(export_pt),task='detect')
ENGINE=Path(exporter.export(format='engine',imgsz=640,batch=1,device=0,
    half=True,dynamic=False,workspace=2,nms=False,simplify=True,opset=17))
torch.cuda.synchronize()
build_seconds=time.perf_counter()-started
(SESSION/'engine_export.json').write_text(json.dumps(dict(engine_sha256=sha256(ENGINE),
    source_sha256=sha256(export_pt),build_seconds=build_seconds,precision='FP16',
    scope='vehicle detector only',shape=[1,3,640,640]),indent=2))
del exporter
gc.collect();torch.cuda.empty_cache()
print('Engine siap:',ENGINE,'build seconds:',round(build_seconds,2))


## 6 Jalankan seluruh pipeline

Setiap run menggunakan proses Python baru: model dan state tracker dimulai ulang, sehingga VRAM tidak menumpuk.
Urutan varian dirotasi antar pengulangan. Run yang sudah lengkap tidak ditimpa; rerun sel ini melanjutkan run berikutnya.
Run parsial/gagal harus diperiksa, tidak dihapus otomatis. Buat SESSION baru jika ingin mengulang dari awal.

**Cold full-pipeline FPS** mencakup load model, semua frame, output dan finalisasi.
**Frame median/P95** mencakup decode, detector, tracker, OCR, logika bisnis dan output frame;
30 frame awal dikeluarkan hanya dari ringkasan timing, tetap diproses untuk state/event.
Final drain OCR dan penutupan output tercakup dalam cold wall time.
Sinkronisasi CUDA dipakai pada pengukuran; stage latency yang bersarang tidak boleh dijumlahkan.


In [ ]:
log=SESSION/'execution.log'
for repeat in range(1,REPEATS+1):
    offset=(repeat-1)%len(VARIANTS)
    for name in VARIANTS[offset:]+VARIANTS[:offset]:
        destination=SESSION/name/f'run-{repeat:02d}'
        completed=destination/'performance.json'
        if completed.exists():
            print('Sudah selesai:',name,repeat);continue
        if destination.exists():raise RuntimeError(f'Output parsial perlu diperiksa: {destination}')
        destination.parent.mkdir(parents=True,exist_ok=True)
        cmd=[sys.executable,'-m','tools.rnd_full_pipeline','--root',str(ROOT),
            '--output',str(destination),'--variant',name,'--engine',str(ENGINE),
            '--max-seconds',str(MAX_SECONDS),'--device','cuda:0']
        print('Menjalankan',name,'pengulangan',repeat,flush=True)
        with log.open('a') as handle:
            process=subprocess.Popen(cmd,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                text=True,bufsize=1,env=dict(os.environ,CUBLAS_WORKSPACE_CONFIG=':4096:8',PYTHONUNBUFFERED='1'))
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
        if process.wait()!=0:raise RuntimeError(f'Run gagal: {name}. Lihat execution.log dan run_summary.json.')
        performance=json.loads(completed.read_text())
        if performance['acceptance_issues']:
            raise RuntimeError(f'Run tidak lengkap: {performance["acceptance_issues"]}; periksa output sebelum lanjut.')
print('Semua pipeline selesai')


## 7 Bandingkan kinerja dan hasil bisnis

Semua hitungan berikut adalah output prediksi, bukan akurasi. OCR complete berarti format terbaca penuh,
bukan nomor lambung terverifikasi. Speed valid berarti pengukuran memenuhi aturan pipeline,
bukan jaminan benar terhadap referensi kecepatan eksternal.

Agreement event mencocokkan waktu final-area <= 0,5 detik dan arah yang sama memakai Hungarian assignment.
Tidak mencocokkan track ID antar tracker. Kendaraan berdekatan waktunya tetap perlu review manual.
Event baseline tidak diperlakukan sebagai ground truth. Pencocokan ini hanya dilakukan pada run pertama.


In [ ]:
from tools.rnd_full_pipeline import summarize
comparison=summarize(SESSION)
fields=['variant','run','valid_run','cold_full_pipeline_fps','frame_ms_median','frame_ms_p95',
    'events_csv','speed_valid','overspeed','hull_text_complete','unique_tracks']
display(comparison[fields])
median_path=SESSION/'comparison_median.csv'
if median_path.exists():
    med=pd.read_csv(median_path).set_index('variant')
    if 'pytorch_fp16' in med.index and 'tensorrt_fp16' in med.index:
        print('Speedup cold pipeline TRT FP16 vs PT FP16:',
            med.loc['tensorrt_fp16','cold_full_pipeline_fps']/med.loc['pytorch_fp16','cold_full_pipeline_fps'])
    display(med[[c for c in fields if c in med.columns]])
display(pd.read_csv(SESSION/'event_agreement.csv'))
for name in VARIANTS:
    path=SESSION/name/'run-01/events.csv'
    print('Event:',name)
    display(pd.read_csv(path,keep_default_na=False)[['event_id','track_id','event_timestamp_s',
        'vehicle_class','speed_kmh','measurement_valid','plate_text','plate_text_status','violation_status','notes']])


## 8 Preview pipeline lengkap

Preview empat panel memakai video beranotasi hasil pipeline, mencakup kotak kendaraan, polygon,
kecepatan, overspeed dan OCR pada frame ketika overlay tersedia sesuai aturan proyek.
Identitas OCR lengkap dan foto sumber tetap dapat diperiksa pada CSV, report dan hull_crops.
Video individual 1920 x 1080 disimpan agar teks kecil tidak bergantung pada preview yang diperkecil.
Ringkasan event di bawah setiap panel baru muncul pada frame event difinalisasi menurut trace;
hasil OCR dari masa depan tidak ditampilkan sebelum waktunya.
Encoding preview dilakukan sesudah benchmark, sehingga tidak masuk timing pipeline.


In [ ]:
from tools.rnd_full_pipeline import preview
from IPython.display import Video,display,Markdown,Image
if not shutil.which('ffmpeg'):
    raise RuntimeError('ffmpeg tidak tersedia; pasang sebelum membuat preview.')
preview_path=preview(SESSION)
display(Video(str(preview_path),embed=True,width=1200))
# Pilih report detail satu varian; foto dibaca langsung agar tampil di Colab.
REPORT_VARIANT='bytetrack'
report_dir=SESSION/REPORT_VARIANT/'run-01/report'
track_records=json.loads((report_dir/'tracks.json').read_text())
for record in track_records[:6]:
    display(Markdown(f'Event {record["event_id"]} | ID {record["track_id"]} | '
        f'Hull {record["plate_text"]} | {record["speed_kmh"]} km/h | {record["violation_status"]}'))
    if record.get('photo'):display(Image(filename=str(report_dir/record['photo']),width=640))


## 9 Unduh seluruh hasil

ZIP hasil berisi lima varian, semua pengulangan, laporan per kendaraan, event CSV, evidence OCR,
video individual, preview empat panel, latency per frame/tahap, metadata dan hash input/source.
Template ground truth dibiarkan kosong agar prediksi tidak dianggap label benar.
HOTA, IDF1 dan IDSW memerlukan anotasi kotak/identitas; akurasi OCR, kecepatan dan event juga perlu ground truth.

Untuk studi lanjutan confidence rendah, ubah konfigurasi kamera pada salinan aset dan
buat SESSION baru untuk SEMUA varian; jangan membandingkan langsung threshold 0,10 dengan baseline 0,25
sebagai efek tracker atau backend. INT8 memerlukan kalibrasi hauling terpisah dari test.


In [ ]:
from google.colab import files
archive=shutil.make_archive(str(SESSION),'zip',SESSION)
print('ZIP hasil:',archive)
files.download(archive)
# Opsional simpan hasil ke Drive setelah seluruh timing selesai:
# shutil.copy2(archive,'/content/drive/MyDrive/'+Path(archive).name)
# Engine tidak otomatis diunduh; build ulang pada GPU/platform/versi TensorRT deployment.


## Referensi dan batas penggunaan

- Kode bisnis proyek hauling: snapshot dan hash pada `code_manifest.json`.
- [Ultralytics v8.3.228](https://github.com/ultralytics/ultralytics/tree/v8.3.228)
- [DeepSORT realtime v1.3.2](https://github.com/levan92/deep_sort_realtime/tree/v1.3.2)
- [MobileNet embedder dan sumber bobot ImageNet](https://github.com/levan92/deep_sort_realtime/blob/v1.3.2/deep_sort_realtime/embedder/embedder_pytorch.py)
- [TensorRT 10.x compatibility dan portability](https://docs.nvidia.com/deeplearning/tensorrt/10.x.x/getting-started/support-matrix.html)
- [TensorRT performance measurement](https://docs.nvidia.com/deeplearning/tensorrt/10.x.x/performance/best-practices.html)

Perubahan GPU, dependency, confidence, shape atau sumber model menghasilkan eksperimen berbeda.
Peringatan/galat OCR tidak boleh diabaikan saat menyimpulkan perbandingan pipeline lengkap.
Hasil cold/steady pipeline Colab juga dipengaruhi CPU host dan video IO, bukan GPU saja.
